# Step 0: Initialization

Download necessary libraries

In [ ]:
import sys

%conda install -c numpy pandas
%conda install -c rasterio
%conda install -c requests
%conda install -c pyproj


Note: you may need to restart the kernel to use updated packages.


'C:\Users\Hangyul' is not recognized as an internal or external command,
operable program or batch file.



Note: you may need to restart the kernel to use updated packages.


'C:\Users\Hangyul' is not recognized as an internal or external command,
operable program or batch file.



Note: you may need to restart the kernel to use updated packages.


'C:\Users\Hangyul' is not recognized as an internal or external command,
operable program or batch file.



Note: you may need to restart the kernel to use updated packages.


'C:\Users\Hangyul' is not recognized as an internal or external command,
operable program or batch file.


# STEP 1: CLEANING THE DATASETS

In [ ]:
# ORIGINAL UNOPTIMIZED VERSION OF CODE

import pandas as pd
import requests

metrics = ["RH2M","WS2M","T2M","T2M_MAX","T2M_MIN","WS50M","ALLSKY_SFC_LW_DWN","WD2M","WD50M","ALLSKY_SFC_SW_DWN"]
lat_min = 32.5
lat_max = 42.1
lng_windows = [(-124.4, -124.4+8), (-124.4+8, -114.1)]

data_years = [2016, 2017, 2021]
cols = ["date","lat","lng"] + metrics

for year in data_years:

    year_data = pd.DataFrame(columns=cols)

    start_time = f'{year}0101'
    end_time = f'{year}1231'
    for lng_range in lng_windows:
        lng_min, lng_max = lng_range[0], lng_range[1]

        lng_range_df = pd.DataFrame(columns=cols)

        for metric in metrics:

            print(f"API URL: {lng_min} - {lng_max} for year {year}")

            nasa_api_url = f'https://power.larc.nasa.gov/api/temporal/daily/regional?start={start_time}&end={end_time}&latitude-min={lat_min}&latitude-max={lat_max}&longitude-min={lng_min}&longitude-max={lng_max}&community=ag&parameters={metric}&format={"json"}&units=imperial&header=false&time-standard=utc'

            response = requests.get(nasa_api_url)
            response.raise_for_status()

            data = response.json()
            data = data['features']

            for point in data:

                location = point['geometry']['coordinates']

                for date in point['properties']['parameter'][metric]:

                    #print(f"{date} at ({location[1]}, {location[0]}) = {point['properties']['parameter'][metric][date]}")
                    expected_row = {
                        "date": date,
                        "lat": location[1],
                        "lng": location[0]
                    }

                    has_row = ((lng_range_df['date'] == date) & (lng_range_df['lat'] == expected_row['lat']) & (lng_range_df['lng'] == expected_row['lng'])).any()
                    if not has_row:
                        lng_range_df = pd.concat([lng_range_df, pd.DataFrame([expected_row])], ignore_index=True)

                    lng_range_df['date'] = date
                    lng_range_df['lat'] = location[1]
                    lng_range_df['lng'] = location[0]
                    lng_range_df[metric] = point['properties']['parameter'][metric][date]

        year_data = pd.concat([year_data, lng_range_df], ignore_index=True)

    year_data.to_csv(f"{year}_weather.csv")
    break

API URL: -124.4 - -116.4 for year 2016


KeyboardInterrupt: 

In [52]:
# OPTIMIZED WITH HELP FROM GEMINI
from concurrent.futures import ThreadPoolExecutor
from itertools import product
import pandas as pd
import requests

session = requests.Session()

metrics = ["RH2M","WS2M","T2M","T2M_MAX","T2M_MIN","WS50M","ALLSKY_SFC_LW_DWN","WD2M","WD50M","ALLSKY_SFC_SW_DWN"]
lat_min = 32.5
lat_max = 42.1
lng_windows = [(-124.4, -124.4+8), (-124.4+8, -114.1)]

data_years = [2016, 2017, 2021]
cols = ["date","lat","lng"] + metrics


# Optimized with gemini
def fetch_data_per_metric(metric, lng_min, lng_max, start_time, end_time):
    lat_range = (32.5, 42.1)
    nasa_api_url = f'https://power.larc.nasa.gov/api/temporal/daily/regional?start={start_time}&end={end_time}&latitude-min={lat_range[0]}&latitude-max={lat_range[1]}&longitude-min={lng_min}&longitude-max={lng_max}&community=ag&parameters={metric}&format={"json"}&units=imperial&header=false&time-standard=utc'

    response = session.get(nasa_api_url)
    response.raise_for_status()
    features = response.json()['features']

    rows = [
        (date, point["geometry"]["coordinates"][1], point["geometry"]["coordinates"][0], val)
        for point in features
        for date, val in point["properties"]["parameter"][metric].items()
    ]

    df = pd.DataFrame(rows, columns=["date", "lat", "lng", metric])
    return df.set_index(["date", "lat", "lng"])[metric]

def construct_data_csv(year):
    tasks = [
        (metric, lng_min, lng_max, start_time, end_time)
        for metric, (lng_min, lng_max), (start_time, end_time)
        in product(metrics, lng_windows, [(f'{year}0101', f'{year}1231')])
    ]

    print(tasks)
    with ThreadPoolExecutor(max_workers=4) as executor:
        results = list(executor.map(lambda args: fetch_data_per_metric(*args), tasks))

    by_metric = {} 
    for series in results:
        metric = series.name
        by_metric.setdefault(metric, []).append(series)


    metric_cols = [pd.concat(series_chunks, axis=0) for series_chunks in by_metric.values()]
    final__df = pd.concat(metric_cols, axis=1).reset_index()

    final__df.head()
    final__df.to_csv(f"{year}_weather.csv")

In [53]:
for year in data_years:
    construct_data_csv(year)

[('RH2M', -124.4, -116.4, '20160101', '20161231'), ('RH2M', -116.4, -114.1, '20160101', '20161231'), ('WS2M', -124.4, -116.4, '20160101', '20161231'), ('WS2M', -116.4, -114.1, '20160101', '20161231'), ('T2M', -124.4, -116.4, '20160101', '20161231'), ('T2M', -116.4, -114.1, '20160101', '20161231'), ('T2M_MAX', -124.4, -116.4, '20160101', '20161231'), ('T2M_MAX', -116.4, -114.1, '20160101', '20161231'), ('T2M_MIN', -124.4, -116.4, '20160101', '20161231'), ('T2M_MIN', -116.4, -114.1, '20160101', '20161231'), ('WS50M', -124.4, -116.4, '20160101', '20161231'), ('WS50M', -116.4, -114.1, '20160101', '20161231'), ('ALLSKY_SFC_LW_DWN', -124.4, -116.4, '20160101', '20161231'), ('ALLSKY_SFC_LW_DWN', -116.4, -114.1, '20160101', '20161231'), ('WD2M', -124.4, -116.4, '20160101', '20161231'), ('WD2M', -116.4, -114.1, '20160101', '20161231'), ('WD50M', -124.4, -116.4, '20160101', '20161231'), ('WD50M', -116.4, -114.1, '20160101', '20161231'), ('ALLSKY_SFC_SW_DWN', -124.4, -116.4, '20160101', '20161231

In [ ]:
def get_clean_csv(csv_path) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    df = df.drop(columns=["ALLSKY_SFC_LW_DWN", "ALLSKY_SFC_SW_DWN", "date", "lat", "lng", "Unnamed: 0"]).dropna()
    df.to_csv(f'{year}_weather_CLEAN.csv')
    return df

for year in data_years:
    csv = get_clean_csv(year)
    print(f"=== DATA FOR {year} ===")
    print(csv.head())

In [106]:
for year in data_years:
    csv = get_clean_csv(year)
    print(f"=== DATA FOR {year} ===")
    print(csv.head())

=== DATA FOR 2016 ===
    RH2M   WS2M    T2M  T2M_MAX  T2M_MIN  WS50M   WD2M  WD50M
0  74.56   5.53  57.22    57.60    56.79   6.24  287.9  289.0
1  78.17  11.30  58.14    60.57    56.66  13.91  177.5  178.2
2  87.55  17.34  61.11    61.59    60.13  23.93  187.3  186.9
3  78.46  13.78  60.96    61.77    60.30  18.36  265.3  265.0
4  79.16  19.35  60.48    61.29    59.74  26.75  272.3  272.6
=== DATA FOR 2017 ===
    RH2M   WS2M    T2M  T2M_MAX  T2M_MIN  WS50M   WD2M  WD50M
0  75.64  16.75  56.97    58.30    56.28  21.79  332.8  332.8
1  77.94   5.55  57.24    58.46    55.71   6.38  327.2  325.9
2  84.27   9.95  59.50    60.42    57.74  13.29  179.9  179.3
3  94.51   9.93  60.15    60.62    59.76  15.08  197.2  198.4
4  97.76   8.34  59.88    60.12    59.02  14.14  266.3  268.0
=== DATA FOR 2021 ===
    RH2M   WS2M    T2M  T2M_MAX  T2M_MIN  WS50M   WD2M  WD50M
0  76.17  13.91  58.30    58.73    58.03  17.92  344.6  344.8
1  85.14   8.59  58.71    59.11    58.30  10.63  297.7  298.4
2  9

In [20]:
import pandas as pd
from osgeo import gdal

gdal.UseExceptions()

def get_crops_for_tif(year) -> pd.DataFrame:
    ds = gdal.Open(f"cropland_data_{year}/{year}_30m_cdls.tif")
    band = ds.GetRasterBand(1)
    rat = band.GetDefaultRAT()

    if rat is not None:
        cols = [rat.GetNameOfCol(i) for i in range(rat.GetColumnCount())]
        data = []
        for r in range(rat.GetRowCount()):
            row_data = []
            for c in range(rat.GetColumnCount()):
                col_type = rat.GetTypeOfCol(c)
                if col_type == gdal.GFT_Integer:
                    row_data.append(rat.GetValueAsInt(r, c))
                elif col_type == gdal.GFT_Real:
                    row_data.append(rat.GetValueAsDouble(r, c))
                else:
                    row_data.append(rat.GetValueAsString(r, c))
            data.append(row_data)
        rat_df = pd.DataFrame(data, columns=cols)
        rat_df['crop_int'] = rat_df.index

        #print(rat_df.head())
        return rat_df.replace({"Class_Names": {"":"UNKNOWN"}}).drop(columns=["Histogram"])
    else:
        print("No embedded Raster Attribute Table")
        return pd.DataFrame()


data_years = [2016, 2017, 2021]
all_crops = [get_crops_for_tif(year) for year in data_years]

crops_df = pd.concat(all_crops, ignore_index=True).drop_duplicates().sort_values('crop_int').reset_index(drop=True)
print(crops_df)
crops_df.to_csv("possible_crops.csv")

                  Class_Names  crop_int
0                  Background         0
1                        Corn         1
2                      Cotton         2
3                        Rice         3
4                     Sorghum         4
..                        ...       ...
251                   UNKNOWN       251
252                   UNKNOWN       252
253                   UNKNOWN       253
254  Dbl Crop Barley/Soybeans       254
255                   UNKNOWN       255

[256 rows x 2 columns]


In [109]:
all_lat_lng_dataframes = []
for year in data_years:
    year_df = pd.read_csv(f'{year}_weather.csv')
    all_lat_lng_dataframes.append(year_df[['lat', 'lng']])

all_lat_lng = pd.concat(all_lat_lng_dataframes, ignore_index=True).drop_duplicates()
all_lat_lng.head()

,lat,lng
0,32.5,-124.375
366,32.5,-123.750
732,32.5,-123.125
1098,32.5,-122.500
1464,32.5,-121.875


In [121]:
from pyproj import Transformer
import rasterio


def get_crop_type_df(year, lat, lng) -> pd.DataFrame:

    with rasterio.open(f'cropland_data_{year}/{year}_30m_cdls.tif') as src:

        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        x, y = transformer.transform(lng, lat)

        values = list(src.sample([(x,y)]))[0]
        #print("Band values: ", values)
        return values[0].item()


get_crop_type_df(2016, 32.5, -124.375)

0

In [2]:
from pyproj import Transformer
import rasterio
import pandas as pd
import numpy as np

data_years = [2016, 2017, 2021]
for year in data_years:
    with rasterio.open(f'cropland_data_{year}/{year}_30m_cdls.tif') as src:
        year_df = pd.read_csv(f'{year}_weather.csv')

        transformer = Transformer.from_crs("EPSG:4326", src.crs, always_xy=True)
        xs, ys = transformer.transform(year_df['lng'].to_numpy(), year_df['lat'].to_numpy())
        coords = zip(xs, ys)
        #print(year_df.columns)

        print(list(src.sample(coords)))
        break
        year_df['crop_int'] = [val[0] for val in src.sample(coords)]

        #year_df.to_csv(f'{year}_with_ys.csv')

        year_df = year_df.drop(columns=["ALLSKY_SFC_LW_DWN", "ALLSKY_SFC_SW_DWN", "date", "lat", "lng", "Unnamed: 0"]).dropna()
        year_df.to_csv(f'{year}_CLEAN.csv')
        

[array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8), array([0], dtype=uint8),

# STEP 2: PREPARING TRAIN/TEST DATA

In [4]:
import pandas as pd
import numpy as np

df_train = pd.DataFrame()
df_test = pd.DataFrame()

for year in data_years:
    df_tmp = pd.read_csv(f'{year}_CLEAN.csv')
    train_split = df_tmp.sample(frac=0.8)

    # Split the data into 80% training data, 80% testing data
    df_train = pd.concat([df_train, train_split], ignore_index=True).drop(columns=["Unnamed: 0"])
    df_test = pd.concat([df_test, df_tmp.drop(train_split.index)], ignore_index=True).drop(columns=["Unnamed: 0"])

# randomize the contents of both the test and training data.
df_test = df_test.sample(frac=1).reset_index()
df_test.to_csv('TEST_DATA.csv')
display(df_test.head())

df_train = df_train.sample(frac=1).reset_index()
df_train.to_csv('TRAIN_DATA.csv')
display(df_train.head())

,index,RH2M,WS2M,T2M,T2M_MAX,T2M_MIN,WS50M,WD2M,WD50M,crop_int
0,3902,96.36,9.73,62.42,62.73,62.17,12.93,300.4,300.2,0
1,19342,22.41,4.05,73.71,87.53,64.54,7.58,48.8,40.0,131
2,66436,45.67,5.57,52.02,67.32,39.78,10.74,65.4,51.8,152
3,72771,32.08,3.18,49.23,67.19,38.91,6.11,54.1,47.2,152
4,34739,95.11,5.55,44.65,52.68,39.54,14.90,159.7,161.7,176


,index,RH2M,WS2M,T2M,T2M_MAX,T2M_MIN,WS50M,WD2M,WD50M,crop_int
0,94263,86.08,8.66,62.35,63.03,61.74,11.14,265.8,265.8,0
1,178636,51.97,7.00,68.54,81.27,56.17,13.38,222.0,219.5,152
2,238264,79.46,15.43,56.41,56.98,55.92,19.91,297.5,297.9,0
3,142197,94.14,9.15,58.37,59.11,57.74,11.07,298.2,298.6,0
4,170283,91.62,10.54,60.51,60.89,60.10,12.97,310.3,311.1,0


# STEP 2: TRAINING THE MODEL

In [21]:
crop_names = pd.read_csv("possible_crops.csv")
print(crop_names.iloc[3]["Class_Names"])
print(len(crop_names))

Rice
256


In [22]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math

import pandas as pd
import numpy as np

df_train = pd.read_csv('TRAIN_DATA.csv')
df_test = pd.read_csv('TEST_DATA.csv')

device = torch.device('cuda' if torch.backends.cuda.is_built() else 'cpu')
print(f'Current device: {device}')

num_epochs = 35
n_batch = 10
worker_threads = 0

evaluated_metrics = ['RH2M','WS2M','T2M','T2M_MAX','T2M_MIN','WS50M','WD2M','WD50M']
num_inputs = len(evaluated_metrics)

crop_names = pd.read_csv("possible_crops.csv")
num_outputs = len(crop_names)

print(f"Model must generate {num_outputs} predictions from {num_inputs} values")


Current device: cuda
Model must generate 256 predictions from 8 values


In [ ]:
# GEMINI CONSULTED FOR ADVICE ON MODEL SETUP

import torch.nn.functional as F

raw_to_contiguous = {raw_id: idx for idx, raw_id in enumerate(crop_names.index)}

class DataViewer(Dataset):
    def __init__(self, in_data: pd.DataFrame):
        self.features = in_data[evaluated_metrics].values.astype("float32")
        mapped_targets = in_data["crop_int"].map(raw_to_contiguous)
        if mapped_targets.isna().any():
            missing = in_data.loc[mapped_targets.isna(), "crop_int"].unique()
            raise ValueError(f"Found crop_int values not possible in possbile_crops.csv")

        self.targets = mapped_targets.values.astype("int64")

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):

        x = torch.tensor(self.features[index], dtype=torch.float32)
        y = torch.tensor(self.targets[index], dtype=torch.int64)
        return x, y

class PredictorModel(nn.Module):
    def __init__(self, num_outputs: int, num_inputs: int):

        super(PredictorModel, self).__init__()

        hidden_1 = 64
        hidden_2 = 128

        self.net = nn.Sequential(
            nn.Linear(num_inputs, hidden_1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_2, num_outputs)
        )
        

    def forward(self, x):

        return self.net(x)

In [ ]:
train_data = DataViewer(df_train)
test_data = DataViewer(df_test)

train_loader = DataLoader(train_data, batch_size=n_batch, num_workers=worker_threads)
test_loader = DataLoader(test_data, batch_size=n_batch, num_workers=worker_threads)

In [ ]:
# NEURAL NETWORK TRAINING REGIME DEVISED BY SELF, DOUBLE-CHECKED BY GEMINI

network = PredictorModel(num_inputs=num_inputs, num_outputs=num_outputs)
optimizer = torch.optim.Adam(network.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

best_performance = math.inf

train_losses_per_epoch = []
valid_losses_per_epoch = []

# Set the network to training mode
print("Begin training....")
network.to(device)
for epoch in range(num_epochs):

    train_loss = 0

    # TRAIN THE MODEL
    network.train()
    for features, labels in train_loader:

        optimizer.zero_grad()

        tensor_x = features.to(device)
        tensor_y = labels.to(device).long()

        pred = network(tensor_x)

        loss = criterion(pred, tensor_y)
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * features.size(0)
    train_loss /= len(train_loader.dataset)
    train_losses_per_epoch.append(train_loss)

    # EVALUATE THE MODEL
    valid_loss = 0
    network.eval()
    with torch.no_grad():
        for features, label in test_loader:

            tensor_x = features.to(device)
            tensor_y = label.to(device)

            pred = network(tensor_x)
            loss = criterion(pred, tensor_y)

            valid_loss += loss.item() * features.size(0)
    valid_loss /= len(test_loader.dataset)
    valid_losses_per_epoch.append(valid_loss)

    # save the model whenevr we do the best.
    if valid_loss < best_performance:
        print(f'>>> EPOCH {epoch} gave a validation loss of {valid_loss} < {best_performance}')
        torch.save(network.state_dict(), f'best_model.pt')
        best_performance = valid_loss

Begin training....
>>> EPOCH 0 gave a validation loss of 0.8759953682392855 < inf
>>> EPOCH 1 gave a validation loss of 0.8536640201893604 < 0.8759953682392855
>>> EPOCH 2 gave a validation loss of 0.8481349191324887 < 0.8536640201893604
>>> EPOCH 3 gave a validation loss of 0.8435554601303664 < 0.8481349191324887
>>> EPOCH 4 gave a validation loss of 0.8432861270085888 < 0.8435554601303664
>>> EPOCH 6 gave a validation loss of 0.8392365040911778 < 0.8432861270085888
>>> EPOCH 12 gave a validation loss of 0.8376719826251987 < 0.8392365040911778


# STEP 3: DEMO

In [66]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
import math

import pandas as pd
import numpy as np


crop_names = pd.read_csv("possible_crops.csv")
raw_to_contiguous = {raw_id: idx for idx, raw_id in enumerate(crop_names.index)}

class DataViewer(Dataset):
    def __init__(self, in_data: pd.DataFrame):
        self.features = in_data[evaluated_metrics].values.astype("float32")
        mapped_targets = in_data["crop_int"].map(raw_to_contiguous)
        if mapped_targets.isna().any():
            missing = in_data.loc[mapped_targets.isna(), "crop_int"].unique()
            raise ValueError(f"Found crop_int values not possible in possbile_crops.csv")

        self.targets = mapped_targets.values.astype("int64")

    def __len__(self):
        return len(self.targets)

    def __getitem__(self, index):

        x = torch.tensor(self.features[index], dtype=torch.float32)
        y = torch.tensor(self.targets[index], dtype=torch.int64)
        return x, y

class PredictorModel(nn.Module):
    def __init__(self, num_outputs: int, num_inputs: int):

        super(PredictorModel, self).__init__()

        hidden_1 = 64
        hidden_2 = 128

        self.net = nn.Sequential(
            nn.Linear(num_inputs, hidden_1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Linear(hidden_1, hidden_2),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(hidden_2, num_outputs)
        )
        

    def forward(self, x):

        return self.net(x)

In [ ]:
import geocoder
import requests
import json

evaluated_metrics = ['RH2M','WS2M','T2M','T2M_MAX','T2M_MIN','WS50M','WD2M','WD50M']

model = PredictorModel(num_outputs=256, num_inputs=len(evaluated_metrics))
model.load_state_dict(torch.load("best_model.pt", map_location=device, weights_only=True))

model.eval()

desired_date = 20261114

g = geocoder.ip('me')
lat = g.latlng[0]
lng = g.latlng[1]

nasa_api_url = f"https://power.larc.nasa.gov/api/temporal/daily/point?start={desired_date}&end={desired_date}&latitude={lat}&longitude={lng}&community=ag&parameters=RH2M%2CWS2M%2CT2M%2CT2M_MAX%2CT2M_MIN%2CWS50M%2CALLSKY_SFC_LW_DWN%2CWD2M%2CWD50M%2CALLSKY_SFC_SW_DWN&format=json&units=metric&header=false&time-standard=utc"
print(nasa_api_url)
response = requests.get(nasa_api_url)
response.raise_for_status()

my_info = response.json()['properties']['parameter']
data = {
    metric: my_info[metric][str(desired_date)] for metric in evaluated_metrics
}

data = pd.DataFrame(data, index=[0])

#f"https://power.larc.nasa.gov/api/temporal/daily/point?start={desired_date}&end={desired_date}&latitude={lat}&longitude={lng}&community=ag&parameters=RH2M%2CWS2M%2CT2M%2CT2M_MAX%2CT2M_MIN%2CWS50M%2CALLSKY_SFC_LW_DWN%2CWD2M%2CWD50M%2CALLSKY_SFC_SW_DWN&format={"json"}&units=metric&header=false&time-standard=utc"
#model()

https://power.larc.nasa.gov/api/temporal/daily/point?start=20261114&end=20261114&latitude=38.1074&longitude=-122.5697&community=ag&parameters=RH2M%2CWS2M%2CT2M%2CT2M_MAX%2CT2M_MIN%2CWS50M%2CALLSKY_SFC_LW_DWN%2CWD2M%2CWD50M%2CALLSKY_SFC_SW_DWN&format=json&units=metric&header=false&time-standard=utc


In [68]:
data = torch.tensor(data.to_numpy(), dtype=torch.float32)

In [ ]:
tensor_x = data.to(device)
model.to(device)
tensor_y = model(tensor_x)
tensor_y.to('cpu')

tensor([[ -9.2513, -15.2303, -18.3108, -22.5798, -22.5222, -22.5885, -22.5611,
         -22.5459, -22.5124, -22.5405, -22.5424, -22.5885, -22.5504, -22.5156,
         -22.5214, -22.5759, -22.5015, -22.4914, -22.5432, -22.5413, -22.5882,
         -22.5562, -22.5460, -22.5350, -20.6573, -22.5687, -22.5580,  -7.5026,
         -28.4473, -22.5854, -22.5246, -22.5747, -22.5318, -22.5164, -22.5622,
         -22.5836,  -5.6863, -22.5205, -22.5807, -22.5147, -22.5440, -22.5338,
         -22.5538, -22.5185, -22.5670, -22.5409, -22.5605, -22.5856, -22.5006,
         -22.5611, -22.5786, -22.5213, -22.5509, -22.5213, -18.0090, -22.5469,
         -22.5852, -22.5504, -22.5796, -22.5698, -22.5675, -14.5843, -22.5438,
         -22.5465, -22.5215, -22.5664, -22.5299, -22.5824, -22.5444, -12.4613,
         -22.5439, -22.5671, -22.5170, -22.5887, -22.5763, -12.2531, -19.3855,
         -22.5721, -22.4995, -22.5974, -22.5433, -22.5151, -22.5386, -22.5920,
         -22.5534, -22.5639, -22.5169, -22.5482, -22

In [ ]:
outputs = tensor_y.tolist()

#print(outputs)

output_list = [(crop_names[crop_names['crop_int'] == i]['Class_Names'].item(), output) for i, output in enumerate(outputs[0])]
output_list = sorted(output_list, key=lambda item: item[1], reverse=True)
print(output_list)

'''
output_dict = {
    crop_names[crop_names['crop_int'] == i]['Class_Names'].item(): output for i, output in enumerate(outputs[0])
}

max_val = max(output_dict.values())
min_val = min(output_dict.values())
if max_val == min_val:
    normalized = {k: 0.0 for k in output_dict}
else:
    normalized = {k: (v - min_val) / (max_val - min_val) for k, v in output_dict.items()}

output_dict = dict(sorted(normalized.items(), key=lambda item: item[1], reverse=True))
print(output_dict)'''

[('Shrubland', 0.3467632532119751), ('Evergreen Forest', -1.1327548027038574), ('Barren', -2.5613129138946533), ('Grassland/Pasture', -2.892876625061035), ('Open Water', -4.168630123138428), ('Alfalfa', -5.686347961425781), ('Woody Wetlands', -6.035953521728516), ('Developed/Low Intensity', -6.86730432510376), ('Rye', -7.502552032470703), ('Background', -9.251341819763184), ('Perennial Ice/Snow ', -10.237638473510742), ('Mixed Forest', -10.746403694152832), ('Almonds', -12.25313949584961), ('Deciduous Forest', -12.418195724487305), ('Grapes', -12.461324691772461), ('Developed/Open Space', -12.6145658493042), ('Developed/Med Intensity', -13.731646537780762), ('Fallow/Idle Cropland', -14.584317207336426), ('Corn', -15.230344772338867), ('Tomatoes', -18.00897789001465), ('Cotton', -18.310768127441406), ('Pistachios', -18.553768157958984), ('Developed/High Intensity', -18.662109375), ('Walnuts', -19.38546371459961), ('Dbl Crop WinWht/Corn', -20.050045013427734), ('Winter Wheat', -20.657327

"\noutput_dict = {\n    crop_names[crop_names['crop_int'] == i]['Class_Names'].item(): output for i, output in enumerate(outputs[0])\n}\n\nmax_val = max(output_dict.values())\nmin_val = min(output_dict.values())\nif max_val == min_val:\n    normalized = {k: 0.0 for k in output_dict}\nelse:\n    normalized = {k: (v - min_val) / (max_val - min_val) for k, v in output_dict.items()}\n\noutput_dict = dict(sorted(normalized.items(), key=lambda item: item[1], reverse=True))\nprint(output_dict)"